# TMDB Batch Data Ingestion

This notebook handles large-scale ingestion of Movie and TV Show metadata
from TMDB Details API into PostgreSQL Bronze layer.

In [17]:
import importlib
import common

importlib.reload(common)

from common import *

engine = get_engine()

from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter

print("Database connection ready.")

Database connection ready.


# TMDB configuration

In [18]:
base_url ="https://api.themoviedb.org/3"
movie_export_url = "https://files.tmdb.org/p/exports/movie_ids_{date}.json.gz"
tv_export_url = "https://files.tmdb.org/p/exports/tv_series_ids_{date}.json.gz"

In [19]:
export_date = "11_19_2026"
export_date_db = "2026-11-19"

tmdb_timeout = (10, 30)

print(
    f"TMDB token: "
    f"{'confirmed' if TMDB_API_TOKEN else 'missing'}"
)

TMDB token: confirmed


# Performance

In [20]:
batch_size = 1000

max_workers = 20
request_per_sec = 25

max_retries = 5

print(f"Batch size: {batch_size}")
print(f"Workers: {max_retries}")
print(f"Rate limit: {request_per_sec} requests/sec")

Batch size: 1000
Workers: 5
Rate limit: 25 requests/sec


# Exporting with URL"s

In [21]:
movie_export_url = movie_export_url.format(date=export_date)

tv_export_url = tv_export_url.format(date=export_date)

In [22]:
def load_export_ids(url):

    ids = set()

    total = 0
    missing = 0
    duplicates = 0

    with requests.get(
        url,
        stream=True,
        timeout=tmdb_timeout
    ) as response:

        response.raise_for_status()
        response.raw.decode_content = False

        with gzip.GzipFile(
            fileobj=response.raw
        ) as file:

            for line in file:

                record = json.loads(line)
                tmdb_id = record.get("id")

                total += 1

                if tmdb_id is None:
                    missing += 1

                elif tmdb_id in ids:
                    duplicates += 1

                else:
                    ids.add(tmdb_id)

    return ids, total, missing, duplicates

# Movie və TV ID's

In [ ]:
movie_ids, movie_total, movie_missing, movie_duplicates = (load_export_ids(movie_export_url))


print("MOVIES")
print(f"Unique IDs: {len(movie_ids):,}")
print(f"Total records: {movie_total:,}")
print(f"Missing IDs: {movie_missing:,}")
print(f"Duplicates: {movie_duplicates:,}")


MOVIES
Unique IDs: 1,278,958
Total records: 1,278,958
Missing IDs: 0
Duplicates: 0
-----------------------


In [26]:
tv_ids, tv_total, tv_missing, tv_duplicates = (load_export_ids(tv_export_url))


print("TV SHOWS")
print(f"Unique IDs: {len(tv_ids):,}")
print(f"Total records: {tv_total:,}")
print(f"Missing IDs: {tv_missing:,}")
print(f"Duplicates: {tv_duplicates:,}")


TV SHOWS
Unique IDs: 237,134
Total records: 237,134
Missing IDs: 0
Duplicates: 0


# Bronze table

In [27]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS bronze.tmdb_metadata (
            media_type VARCHAR(10) NOT NULL,
            tmdb_id BIGINT NOT NULL,
            payload JSONB NOT NULL,
            ingested_at TIMESTAMPTZ DEFAULT NOW(),
            PRIMARY KEY (media_type, tmdb_id)
        );
    """))

print("bronze.tmdb_metadata ready.")

bronze.tmdb_metadata ready.


In [28]:
with engine.connect() as conn:
    checkpoints = conn.execute(text("""
        SELECT
            media_type,
            export_date,
            last_processed_id,
            records_processed,
            last_processed_at
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND export_date = :export_date
        ORDER BY media_type;
    """), {
        "export_date": export_date_db
    })

    for row in checkpoints:
        print(row)

('movie', datetime.date(2026, 11, 19), 270990, 139993, datetime.datetime(2026, 11, 24, 0, 8, 1, 752595, tzinfo=datetime.timezone.utc))
('tv', datetime.date(2026, 11, 19), None, 0, None)


# data importing

In [29]:
with engine.connect() as conn:

    existing_movie_ids = set(
        conn.execute(text("""
            SELECT tmdb_id
            FROM bronze.tmdb_metadata
            WHERE media_type = 'movie'
        """)).scalars().all()
    )

    existing_tv_ids = set(
        conn.execute(text("""
            SELECT tmdb_id
            FROM bronze.tmdb_metadata
            WHERE media_type = 'tv'
        """)).scalars().all()
    )

print(f"Existing movies: {len(existing_movie_ids):,}")
print(f"Existing TV shows: {len(existing_tv_ids):,}")

Existing movies: 157,993
Existing TV shows: 0


In [ ]:
_thread_local = threading.local()


def get_session():

    if not hasattr(_thread_local, "session"):

        session = requests.Session()

        adapter = HTTPAdapter(
            pool_connections=max_workers,
            pool_maxsize=max_workers
        )

        session.mount(
            "https://",
            adapter
        )

        session.headers.update({
            "Authorization": f"Bearer {TMDB_API_TOKEN}"
        })

        _thread_local.session = session

    return _thread_local.session

In [36]:
rate_lock = threading.Lock()
next_request_time = time.monotonic()


def throttle():

    global next_request_time

    with rate_lock:

        now = time.monotonic()

        wait_time = next_request_time - now

        next_request_time = (
            max(now, next_request_time)
            + 1 / request_per_sec
        )

    if wait_time > 0:
        time.sleep(wait_time)

In [37]:
def get_details(tmdb_id, media_type):

    url = f"{base_url}/{media_type}/{tmdb_id}"

    for attempt in range(max_retries):

        throttle()

        try:

            response = get_session().get(
                url,
                timeout=TMDB_TIMEOUT
            )

            if response.status_code == 200:
                return {
                    "tmdb_id": tmdb_id,
                    "data": response.json(),
                    "error": None
                }

            if response.status_code == 404:
                return {
                    "tmdb_id": tmdb_id,
                    "data": None,
                    "error": None
                }

            if (
                response.status_code == 429
                or response.status_code >= 500
            ):

                retry_after = response.headers.get(
                    "Retry-After"
                )

                if retry_after:
                    wait_time = float(retry_after)
                else:
                    wait_time = min(
                        2 ** attempt,
                        30
                    )

                time.sleep(wait_time)
                continue

            response.raise_for_status()

        except requests.RequestException as error:

            if attempt == max_retries - 1:

                return {
                    "tmdb_id": tmdb_id,
                    "data": None,
                    "error": str(error)
                }

            wait_time = min(
                2 ** attempt,
                30
            )

            time.sleep(wait_time)

    return {
        "tmdb_id": tmdb_id,
        "data": None,
        "error": "Max retries exceeded"
    }

In [40]:
def fetch_batch(batch_ids, media_type):

    results = []

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        futures = {
            executor.submit(
                get_details,
                tmdb_id,
                media_type
            ): tmdb_id
            for tmdb_id in batch_ids
        }

        for future in as_completed(futures):

            result = future.result()
            results.append(result)

    return results

In [ ]:
with engine.connect() as conn:

    last_processed_id = conn.execute(text("""
        SELECT last_processed_id
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'movie'
          AND export_date = :export_date
    """), {
        "export_date": export_date_db
    }).scalar()

new_movie_ids = sorted(
    tmdb_id
    for tmdb_id in movie_ids
    if tmdb_id not in existing_movie_ids
    and (
        last_processed_id is None
        or tmdb_id > last_processed_id
    )
)

print(f"Last processed ID: {last_processed_id}")
print(f"Remaining movies: {len(new_movie_ids):,}")

In [41]:
test_ids = sorted(movie_ids)[:100]

start_time = time.perf_counter()

test_results = fetch_batch(
    test_ids,
    "movie"
)

elapsed = time.perf_counter() - start_time

successful = sum(
    result["data"] is not None
    for result in test_results
)

failed = sum(
    result["error"] is not None
    for result in test_results
)

print(f"IDs tested: {len(test_ids):,}")
print(f"Successful: {successful:,}")
print(f"Failed: {failed:,}")
print(f"Time: {elapsed:.2f} sec")
print(f"Rate: {len(test_ids) / elapsed:.2f} requests/sec")

NameError: name 'MAX_WORKERS' is not defined